In [1]:
import os
from pathlib import Path

BASE_DIR = Path.cwd()
ROOT_DIR = BASE_DIR.parent
CACHE_DIR = str(ROOT_DIR.parent / "huggingface_cache")

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import random
import json
from tqdm import tqdm
import torch
from dotenv import load_dotenv
from transformers import set_seed
from unsloth import FastLanguageModel, get_chat_template
load_dotenv()

SEED = 42
set_seed(SEED)
random.seed(SEED)

MAX_SEQ_LENGTH = 4096

sys.path.insert(0, str(ROOT_DIR))

DATA_UN_LABEL_DIR = (ROOT_DIR / "data/dpo_data/un_label").resolve()
DATA_LABELED_DIR = (ROOT_DIR / "data/dpo_data/labeled/data.json").resolve()
RESULT_DIR = (ROOT_DIR / "data/dpo_data/result_for_unlabel").resolve()
MODEL_DIR = (ROOT_DIR / "models").resolve()

/opt/conda/envs/dopn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/tmp/ipykernel_19796/2795264684.py:18: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, get_chat_template


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

def get_model_and_tokenizer():
    model = AutoModelForCausalLM.from_pretrained(
        "unsloth/Qwen3-8B",
        dtype=torch.bfloat16,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-8B")
    return model, tokenizer

In [7]:
model, tokenizer = get_model_and_tokenizer()
from peft import PeftModel
model = PeftModel.from_pretrained(model, MODEL_DIR / "continual-pretrain")
model = model.merge_and_unload()  # merge vào base model

# Sau đó load sft adapter
model = PeftModel.from_pretrained(model, MODEL_DIR / "sft")

Loading weights: 100%|██████████| 399/399 [00:02<00:00, 155.73it/s]


In [8]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [79]:
mcq_path = DATA_UN_LABEL_DIR / "mcq.json"
qna_path = DATA_UN_LABEL_DIR / "qna.json"

with open(mcq_path, "r", encoding="utf-8") as f:
    mcq_data = json.load(f)

with open(qna_path, "r", encoding="utf-8") as f:
    qna_data = json.load(f)

print(f"MCQ samples: {len(mcq_data)}")
print(f"QnA samples: {len(qna_data)}")

MCQ samples: 2000
QnA samples: 2000


In [80]:
def apply_template(prompt):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    return text

In [81]:
def convert_mcq(sample):
    question = sample["question"]
    choices = sample["choices"]

    formatted = question + "\nChoices:\n"
    for i, c in enumerate(choices, 1):
        formatted += f"{i}. {c}\n"

    formatted += (
        "\nAnswer the question.\n"
        "Format:\n"
        "The answer is <number>, then explain your choice.\n"
    )

    return {
        "prompt": apply_template(formatted.strip()),
        "raw": sample, 
        "type": "mcq"
    }


def convert_qna(sample):
    return {
        "prompt": apply_template(sample["question"]),
        "raw": sample,
        "type": "qna"
    }


mcq_converted = [convert_mcq(x) for x in mcq_data]
qna_converted = [convert_qna(x) for x in qna_data]

all_data = mcq_converted + qna_converted

In [87]:
def generate_batch(texts, max_new_tokens=1024):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    input_lengths = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
        )

    responses = []
    for i in range(len(texts)):
        gen_tokens = outputs[i][input_lengths:]
        text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        responses.append(text.strip())

    return responses

In [86]:
import json
from pathlib import Path
from tqdm import tqdm

MCQ_FILE = RESULT_DIR / "mcq_generated.jsonl"
QNA_FILE = RESULT_DIR / "qna_generated.jsonl"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

for path in [MCQ_FILE, QNA_FILE]:
    if not path.exists():
        path.touch()

def append_jsonl(path, data_list):
    with open(path, "a", encoding="utf-8") as f:
        for item in data_list:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [88]:
def run_batch_generation(all_data, batch_size=4):
    for i in tqdm(range(0, len(all_data), batch_size)):
        batch = all_data[i:i + batch_size]

        texts = [x["prompt"] for x in batch]
        responses = generate_batch(texts)

        mcq_batch = []
        qna_batch = []

        for sample, response in zip(batch, responses):
            item = {
                **sample["raw"],
                "prompt": sample["prompt"],
                "model_response": response,
            }

            if sample["type"] == "mcq":
                mcq_batch.append(item)
            else:
                qna_batch.append(item)

        if mcq_batch:
            append_jsonl(MCQ_FILE, mcq_batch)

        if qna_batch:
            append_jsonl(QNA_FILE, qna_batch)

In [89]:
run_batch_generation(all_data, batch_size=32)

100%|██████████| 125/125 [2:18:09<00:00, 66.32s/it] 


In [94]:
import json
import re
from pathlib import Path

def extract_answer(text):
    match = re.search(r"The answer is\s*<?\s*(\d+)\s*>?", text, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None


MCQ_FILE = Path(MCQ_FILE)
TMP_FILE = MCQ_FILE.with_suffix(".tmp.jsonl")

updated_data = []

with open(MCQ_FILE, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        sample = json.loads(line)

        if idx < 1000:
            sample["lang"] = "en"
        else:
            sample["lang"] = "vi"

        gt = sample.get("answer")
        pred = extract_answer(sample.get("model_response", ""))

        if pred is None:
            sample["is_correct"] = None
        else:
            sample["is_correct"] = (pred == gt)

        updated_data.append(sample)


with open(TMP_FILE, "w", encoding="utf-8") as f:
    for item in updated_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")


TMP_FILE.replace(MCQ_FILE)

PosixPath('/home/jovyan/dopn/AI-Agent-System-in-Telecom/data/dpo_data/result_for_unlabel/mcq_generated.jsonl')

In [ ]:
import json
from pathlib import Path
from bert_score import score

TMP_FILE = QNA_FILE.with_suffix(".tmp.jsonl")

BATCH_SIZE = 32

def compute_and_write(batch, out_file, lang):
    if not batch:
        return

    cands = [x["model_response"] for x in batch]
    refs = [x["answer"] for x in batch]

    P, R, F1 = score(cands, refs, lang=lang, verbose=False)

    with open(out_file, "a", encoding="utf-8") as f:
        for i, sample in enumerate(batch):
            sample["bert_p"] = float(P[i])
            sample["bert_r"] = float(R[i])
            sample["bert_f1"] = float(F1[i])

            f.write(json.dumps(sample, ensure_ascii=False) + "\n")


if TMP_FILE.exists():
    TMP_FILE.unlink()


batch = []
current_lang = None


with open(QNA_FILE, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        sample = json.loads(line)

        cand = sample.get("model_response", "").strip()[:512]
        ref = sample.get("answer", "").strip()[:512]

        lang = "en" if idx < 1000 else "vi"

        sample["lang"] = lang
        sample["idx"] = idx
        sample["model_response"] = cand
        sample["answer"] = ref

        if current_lang is not None and lang != current_lang:
            compute_and_write(batch, TMP_FILE, current_lang)
            batch = []

        current_lang = lang
        batch.append(sample)

        if len(batch) >= BATCH_SIZE:
            compute_and_write(batch, TMP_FILE, current_lang)
            batch = []


compute_and_write(batch, TMP_FILE, current_lang)

TMP_FILE.replace(QNA_FILE)

In [101]:
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

mcq_data = load_jsonl(MCQ_FILE)
qna_data = load_jsonl(QNA_FILE)

In [93]:
def extract_prompt(text: str):
    try:
        start = text.index("<|im_start|>user") + len("<|im_start|>user")
        end = text.index("<|im_end|>", start)
        return text[start:end].strip()
    except:
        return text.strip()

In [106]:
def build_chosen_mcq(sample):
    ans = sample["answer"]
    exp = sample.get("explanation", "")

    if isinstance(sample.get("lang", "en"), str) and sample["lang"] == "vi":
        return f"Đáp án là {ans}. {exp}"
    else:
        return f"The answer is {ans}. {exp}"


def build_chosen_qna(sample):
    return sample["answer"]

In [104]:
dpo = []

for s in tqdm(mcq_data):
    if s.get("is_correct", True):
        continue

    dpo.append({
        "prompt": extract_prompt(s["prompt"]),
        "chosen": build_chosen_mcq(s),
        "rejected": s["model_response"]
    })

100%|██████████| 2000/2000 [00:00<00:00, 638062.52it/s]


In [108]:
BERT_THRESHOLD = 1

for s in tqdm(qna_data):
    if s.get("bert_f1", 1.0) >= BERT_THRESHOLD:
        continue

    dpo.append({
        "prompt": extract_prompt(s["prompt"]),
        "chosen": build_chosen_qna(s),
        "rejected": s["model_response"]
    })

100%|██████████| 2000/2000 [00:00<00:00, 503064.95it/s]


In [111]:
import json

with open(DATA_LABELED_DIR, "w", encoding="utf-8") as f:
    json.dump(dpo, f, ensure_ascii=False, indent=2)

print("DPO samples:", len(dpo))

DPO samples: 2549
